# NWDAF Operations Dashboard - GCP Configuration

This notebook helps you configure your Google Cloud environment so that the **NWDAF Operations Dashboard** can connect to your remote `open5gs-nwdafd` instance without requiring a local SSH tunnel.

### What this does:
1. Authenticates you with Google Cloud.
2. Sets your project and zone.
3. Ensures your 5G AI Lab VM is running.
4. Creates a firewall rule to expose port `7779` (NWDAF REST API) to the internet.
5. Retrieves your VM's External IP and generates the correct **Base URL** for your Dashboard settings.

In [ ]:
# 1. Authenticate to Google Cloud
from google.colab import auth
auth.authenticate_user()
print('✅ Authenticated successfully!')

In [ ]:
# 2. Configure GCP Project and Zone variables
PROJECT_ID = 'g-ai-lab-491619'
ZONE = 'europe-west4-a'
VM_NAME = 'open5gs-ai-lab'

!gcloud config set project {PROJECT_ID}
!gcloud config set compute/zone {ZONE}
print(f'✅ Configured for project {PROJECT_ID} in zone {ZONE}')

In [ ]:
# 3. Check VM Status and Start if necessary
!echo "Checking status of VM: {VM_NAME}..."
status = !gcloud compute instances describe {VM_NAME} --format="value(status)"

if status and status[0] == 'TERMINATED':
    print(f'⏳ VM {VM_NAME} is stopped. Starting it up...')
    !gcloud compute instances start {VM_NAME}
    print('✅ VM started!')
else:
    print(f'✅ VM {VM_NAME} is currently: {status[0] if status else "UNKNOWN"}')

In [ ]:
# 4. Create Firewall Rule to open port 7779
# This allows the React dashboard (running in your browser) to reach the API directly.

FIREWALL_RULE_NAME = 'allow-nwdaf-dashboard-7779'

!echo "Checking for existing firewall rule..."
rule_exists = !gcloud compute firewall-rules list --filter="name={FIREWALL_RULE_NAME}" --format="value(name)"

if not rule_exists:
    print(f'⏳ Creating firewall rule {FIREWALL_RULE_NAME}...')
    !gcloud compute firewall-rules create {FIREWALL_RULE_NAME} \
        --direction=INGRESS \
        --priority=1000 \
        --network=default \
        --action=ALLOW \
        --rules=tcp:7779 \
        --source-ranges=0.0.0.0/0 \
        --target-tags=nwdaf-api
    
    # Apply the network tag to the VM so the firewall rule affects it
    !gcloud compute instances add-tags {VM_NAME} --tags=nwdaf-api
    print('✅ Firewall rule created and applied to VM!')
else:
    print(f'✅ Firewall rule {FIREWALL_RULE_NAME} already exists.')

In [ ]:
# 5. Retrieve External IP and Generate Dashboard URL
external_ip = !gcloud compute instances describe {VM_NAME} --format="get(networkInterfaces[0].accessConfigs[0].natIP)"

if external_ip:
    ip = external_ip[0]
    base_url = f"http://{ip}:7779/nwdaf-analytics/v1"
    
    print("="*60)
    print("🚀 CONFIGURATION COMPLETE")
    print("="*60)
    print("\nPaste the following URL into your Dashboard Settings:\n")
    print(f"   Base URL : {base_url}")
    print("\nConnection Mode: Direct")
    print("="*60)
else:
    print("❌ Could not retrieve External IP. Is the VM running?")

### ⚠️ Important Note regarding CORS (Cross-Origin Resource Sharing)
If you use this Direct IP method, your browser will make cross-origin requests. 
Ensure that your `open5gs-nwdafd` is configured to allow CORS, or you will see network errors in the dashboard.
If CORS is not enabled on the C++ server, you **must** use the SSH Tunnel method described in `nwdaf_dashboard_setup.md`.